# Curvature-Coupled Dark Energy: Metric Response and Linear Power Spectra

This notebook reproduces the linear-perturbation figures used in the
**Curvature-Coupled Dark Energy (CCDE)** paper:

- **Fig. 7:** phenomenological Poisson function $\mu_\Psi(k,z)$ and gravitational slip
  $\gamma(k,z)=\Phi/\Psi$ at $z=0$ and $z=0.5$;
- **Fig. 9:** linear matter power spectrum and dimensionless Weyl-potential spectrum at $z=0$;
- **Fig. 11:** Weyl-potential power-spectrum ratios at $z=0.25,\ 0.75,\ 1,\ 2$.

Only the calculations required for these paper figures are retained.


In [ ]:

import os
from os.path import exists

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from matplotlib.ticker import (
    LogLocator,
    LogFormatterMathtext,
    NullFormatter,
    FixedLocator,
    MultipleLocator,
)


# -------------------------------------------------
# Shared helpers and containers
# -------------------------------------------------
def nested_dict(n, value_type):
    """Create an n-level nested defaultdict-like container."""
    from collections import defaultdict
    if n == 1:
        return defaultdict(value_type)
    return defaultdict(lambda: nested_dict(n - 1, value_type))


data = nested_dict(5, list)
out_arr = nested_dict(5, list)


# -------------------------------------------------
# Paper-wide plotting convention
# -------------------------------------------------
text_size = 30
label_fs = 34
lw_f = 3

plt.rc('text', usetex=True)
plt.rc('font', family='normal', weight='bold', size=text_size)

# Fixed model-to-colour mapping used in the other CCDE notebooks.
colors = [
    '#000000',  # (sigma=0.001, alpha=0.001)
    '#0072B2',  # (sigma=0.3,   alpha=0.1)
    '#E69F00',  # (sigma=0.3,   alpha=0.001)
    '#009E73',  # (sigma=0.3,   alpha=-0.1)
    '#D55E00',  # (sigma=1,     alpha=0.1)
    '#56B4E9',  # (sigma=1,     alpha=-0.1)
    '#CC79A7',  # (sigma=1.5,   alpha=0.1)
    '#F0E442',  # (sigma=1.5,   alpha=0.001)
    '#882255',  # (sigma=1.5,   alpha=-0.05)
]

alpha_consts = [
    "0.001", "0.1", "0.001", "-0.1",
    "0.1", "-0.1", "0.1", "0.001", "-0.05"
]
sigma_consts = [
    "0.001", "0.3", "0.3", "0.3",
    "1.", "1.", "1.5", "1.5", "1.5"
]

major_alpha = 0.35
minor_alpha = 0.15


## 1. Load CCDE background and transfer-function outputs

The notebook keeps the original data structure,

`data["bg_data"][sigma_key][alpha_key]`,  
`data["pk"][z][sigma_key][alpha_key]`, and  
`data["tk"][z][sigma_key][alpha_key]`.

The files are read from the public CCDE transfer-function directory using the renamed
`CCDE_...` convention. A separate $\Lambda$CDM output is loaded for the ratios shown in
paper Figs. 9 and 11.


In [ ]:

# -------------------------------------------------
# Configuration
# -------------------------------------------------
redshifts = [
    100, 50, 30, 20, 10, 6, 5, 4, 3, 2.5, 2, 1.75, 1.5, 1.25, 1.1, 1.0,
    0.9, 0.8, 0.75, 0.7, 0.6, 0.5, 0.4, 0.3, 0.25, 0.2, 0.1, 0.0
]

data_address = "./../DataGenerator/transfer_functions/"
load_data = True

alphas = ["0.1", "0.05", "0.001", "-0.05", "-0.1"]
sigmas = ["0.001", "0.3", "1.", "1.5"]


def skey(s):
    return f"sigma={s}"


def akey(a):
    return f"alpha={a}"


# CLASS output uses z indices in the filenames: z1, z2, ...
z_to_ind = {z: i + 1 for i, z in enumerate(redshifts)}

# Stable indexing for the parameter grid.
alpha = np.array(sorted(set(alphas), key=float), dtype=object)
sigma = np.array(sorted(set(sigmas), key=float), dtype=object)

alpha_to_j = {a: j for j, a in enumerate(alpha)}
sigma_to_i = {s: i for i, s in enumerate(sigma)}

out_arr = np.empty((len(sigma), len(alpha)), dtype=object)

# Preserve the original data construction.
data.setdefault('bg_data', {})
data.setdefault('pk', {})
data.setdefault('tk', {})

for z in redshifts:
    data['pk'].setdefault(z, {})
    data['tk'].setdefault(z, {})

for s in sigma:
    data['bg_data'].setdefault(skey(s), {})
    for a in alpha:
        data['bg_data'][skey(s)].setdefault(akey(a), None)


# -------------------------------------------------
# LambdaCDM reference
# -------------------------------------------------
lcdm_sigma = "0.0"
lcdm_alpha = "0.0"
lcdm_dir = os.path.join(data_address, "LCDM")

lcdm_bg = os.path.join(lcdm_dir, "LCDM_background.dat")

data['bg_data'].setdefault(skey(lcdm_sigma), {})
data['bg_data'][skey(lcdm_sigma)].setdefault(akey(lcdm_alpha), None)

for z in redshifts:
    data['pk'][z].setdefault(skey(lcdm_sigma), {})
    data['pk'][z][skey(lcdm_sigma)].setdefault(akey(lcdm_alpha), None)

    data['tk'][z].setdefault(skey(lcdm_sigma), {})
    data['tk'][z][skey(lcdm_sigma)].setdefault(akey(lcdm_alpha), None)

if exists(lcdm_bg):
    print("\033[94mLoading LambdaCDM reference\033[0m")
    data['bg_data'][skey(lcdm_sigma)][akey(lcdm_alpha)] = np.loadtxt(lcdm_bg)

    for z in redshifts:
        z_ind = z_to_ind[z]

        pk_file = os.path.join(lcdm_dir, f"LCDM_z{z_ind}_pk.dat")
        tk_file = os.path.join(lcdm_dir, f"LCDM_z{z_ind}_tk.dat")

        if exists(pk_file):
            data['pk'][z][skey(lcdm_sigma)][akey(lcdm_alpha)] = np.loadtxt(pk_file)

        if exists(tk_file):
            data['tk'][z][skey(lcdm_sigma)][akey(lcdm_alpha)] = np.loadtxt(tk_file)
else:
    print(lcdm_bg, "\033[91mLambdaCDM background file not found.\033[0m")


# -------------------------------------------------
# CCDE parameter grid
# -------------------------------------------------
total_loaded = 0
missing_bg = []
missing_pk = 0
missing_tk = 0

if load_data:
    for a_str in alpha:
        for s_str in sigma:

            output_dir = f"sigma{s_str}_alpha{a_str}"
            base_path = os.path.join(data_address, "run_" + output_dir)

            bg_file = os.path.join(
                base_path,
                f"CCDE_sigma{s_str}_alpha{a_str}_background.dat"
            )

            i = sigma_to_i[s_str]
            j = alpha_to_j[a_str]
            out_arr[i][j] = output_dir

            data['bg_data'].setdefault(skey(s_str), {})
            data['bg_data'][skey(s_str)].setdefault(akey(a_str), None)

            for z in redshifts:
                data['pk'][z].setdefault(skey(s_str), {})
                data['pk'][z][skey(s_str)].setdefault(akey(a_str), None)

                data['tk'][z].setdefault(skey(s_str), {})
                data['tk'][z][skey(s_str)].setdefault(akey(a_str), None)

            if not exists(bg_file):
                missing_bg.append(bg_file)
                continue

            print(f"\033[94mLoading {output_dir}\033[0m")
            data['bg_data'][skey(s_str)][akey(a_str)] = np.loadtxt(bg_file)
            total_loaded += 1

            for z in redshifts:
                z_ind = z_to_ind[z]

                pk_file = os.path.join(
                    base_path,
                    f"CCDE_sigma{s_str}_alpha{a_str}_z{z_ind}_pk.dat"
                )
                tk_file = os.path.join(
                    base_path,
                    f"CCDE_sigma{s_str}_alpha{a_str}_z{z_ind}_tk.dat"
                )

                if exists(pk_file):
                    data['pk'][z][skey(s_str)][akey(a_str)] = np.loadtxt(pk_file)
                else:
                    missing_pk += 1

                if exists(tk_file):
                    data['tk'][z][skey(s_str)][akey(a_str)] = np.loadtxt(tk_file)
                else:
                    missing_tk += 1

print("Number of CCDE simulations loaded:", total_loaded)
print("Alphas loaded:", list(alpha))
print("Sigmas loaded:", list(sigma))

if missing_bg:
    print(f"\033[93mMissing {len(missing_bg)} background files.\033[0m")

if missing_pk or missing_tk:
    print(
        f"\033[93mMissing spectra counts: pk={missing_pk}, tk={missing_tk}\033[0m"
    )


## 2. Phenomenological Poisson and slip functions — paper Fig. 7

The gravitational slip is
$
\gamma(k,z)=\frac{\Phi}{\Psi},
$
while the effective Poisson function is defined by
$
-k^2\Psi
=
4\pi G_{\rm N}a^2\mu_\Psi(k,z)\rho_{\rm m}\Delta_{\rm m}.
$

The figure is evaluated at $z=0$ and $z=0.5$. The transfer-function wavenumber is converted
from $h\,{\rm Mpc}^{-1}$ to ${\rm Mpc}^{-1}$ inside the Poisson equation. At these late
redshifts, the total velocity transfer function is matter dominated and is used in the
comoving-density correction.


In [ ]:

# -------------------------------------------------
# Transfer/background column maps
# -------------------------------------------------
COL = dict(
    k=0,
    d_g=1,
    d_b=2,
    d_cdm=3,
    d_ur=4,
    d_ncdm=5,
    d_m=8,
    d_tot=9,
    phi=10,
    psi=11,
    t_g=18,
    t_b=19,
    t_ur=20,
    t_ncdm=21,
    t_tot=22,
)

BG = dict(
    z=0,
    H=3,
    rho_g=8,
    rho_b=9,
    rho_cdm=10,
    rho_ncdm=11,
    rho_ur=13,
)

h = 0.6688
z_plots = [0.0, 0.5]


def require_model(data, z, sig_key, alp_key):
    if 'tk' not in data or z not in data['tk']:
        raise KeyError(f"Missing data['tk'][{z}].")
    if sig_key not in data['tk'][z] or alp_key not in data['tk'][z][sig_key]:
        raise KeyError(f"Missing tk for {sig_key}, {alp_key} at z={z}.")
    if (
        'bg_data' not in data
        or sig_key not in data['bg_data']
        or alp_key not in data['bg_data'][sig_key]
    ):
        raise KeyError(f"Missing background for {sig_key}, {alp_key}.")


def bg_at_z(bg_arr, z_target):
    """Return scale factor, conformal Hubble rate, and standard-fluid densities."""
    z_bg = bg_arr[:, BG['z']]
    i = np.argmin(np.abs(z_bg - z_target))

    z = z_bg[i]
    a = 1.0 / (1.0 + z)

    # Background output stores physical H; convert to conformal H = a H.
    H_conf = bg_arr[i, BG['H']] / (1.0 + z)

    rho_g = bg_arr[i, BG['rho_g']]
    rho_b = bg_arr[i, BG['rho_b']]
    rho_cdm = bg_arr[i, BG['rho_cdm']]
    rho_ncdm = bg_arr[i, BG['rho_ncdm']]
    rho_ur = bg_arr[i, BG['rho_ur']]

    return a, H_conf, rho_g, rho_b, rho_cdm, rho_ncdm, rho_ur


def compute_delta_components(
    tk_arr,
    rho_g,
    rho_b,
    rho_cdm,
    rho_ncdm,
    rho_ur,
):
    """Construct the late-time matter density contrast from minimally coupled species."""
    d_g = tk_arr[:, COL['d_g']]
    d_b = tk_arr[:, COL['d_b']]
    d_cdm = tk_arr[:, COL['d_cdm']]
    d_ur = tk_arr[:, COL['d_ur']]
    d_ncdm = tk_arr[:, COL['d_ncdm']]

    rho_m = rho_b + rho_cdm + rho_ncdm
    delta_m = (
        rho_b * d_b
        + rho_cdm * d_cdm
        + rho_ncdm * d_ncdm
    ) / rho_m

    rho_tot = rho_g + rho_ur + rho_b + rho_cdm + rho_ncdm
    delta_tot = (
        rho_g * d_g
        + rho_ur * d_ur
        + rho_b * d_b
        + rho_cdm * d_cdm
        + rho_ncdm * d_ncdm
    ) / rho_tot

    return delta_m, delta_tot, rho_m, rho_tot


def compute_Delta_m(tk_arr, H_conf, delta_m):
    """
    Construct the comoving matter perturbation.

    At z=0 and z=0.5 the total standard-fluid velocity is matter dominated,
    so t_tot provides the late-time velocity correction used in the original analysis.
    """
    k_Mpc = tk_arr[:, COL['k']] * h
    theta_tot = tk_arr[:, COL['t_tot']] # 1/Mpc

    Delta_m = delta_m + 3.0 * H_conf * theta_tot / (k_Mpc**2)

    return k_Mpc, theta_tot, Delta_m


def compute_gamma_mu(tk_arr, a, rho_m, Delta_m):
    """
    Compute gamma = Phi/Psi and mu_Psi from the Psi-Poisson equation.
    Background densities follow the CLASS normalization (8 pi G / 3) rho.
    """
    k_Mpc = tk_arr[:, COL['k']] * h

    phi = tk_arr[:, COL['phi']]
    psi = tk_arr[:, COL['psi']]

    eps = 1e-60
    gamma = phi / (psi + eps)

    # Since the stored density is (8 pi G / 3) rho,
    # 4 pi G rho = (3/2) rho_stored.
    mu_psi = (
        -k_Mpc**2 * psi
        / (1.5 * a**2 * rho_m * Delta_m)
    )

    return gamma, mu_psi


# -------------------------------------------------
# Figure: gamma and mu_Psi at z=0 and z=0.5
# -------------------------------------------------
fig, axes = plt.subplots(
    2, 2,
    figsize=(20, 13),
    facecolor='w',
    sharex='all',
)

plt.subplots_adjust(wspace=0.2, hspace=0.0)

ax_g0, ax_mu0 = axes[0]
ax_g1, ax_mu1 = axes[1]

for ax in axes.flat:
    ax.tick_params(which='both', direction='in', top=True, right=True)
    ax.grid(True, which='major', alpha=major_alpha)
    ax.grid(True, which='minor', alpha=minor_alpha)
    ax.minorticks_on()

    ax.set_xscale('log')
    ax.xaxis.set_major_locator(
        LogLocator(base=10.0, subs=(1.0,), numticks=1000)
    )
    ax.xaxis.set_minor_locator(
        LogLocator(
            base=10.0,
            subs=np.arange(2, 10) * 0.1,
            numticks=1000,
        )
    )
    ax.xaxis.set_major_formatter(LogFormatterMathtext())
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.set_xlim(6e-5, 1.0)

for ax in [ax_g0, ax_g1, ax_mu0, ax_mu1]:
    ax.yaxis.set_major_locator(MultipleLocator(0.1))
    ax.yaxis.set_minor_locator(MultipleLocator(0.05))
    ax.axhline(1.0, lw=1.5, c='k', alpha=0.7)

ax_g0.set_ylabel(r'$\gamma(k,z=0)=\Phi/\Psi$', fontsize=label_fs)
ax_mu0.set_ylabel(r'$\mu_{\Psi}(k,z=0)$', fontsize=label_fs)
ax_g1.set_ylabel(r'$\gamma(k,z=0.5)=\Phi/\Psi$', fontsize=label_fs)
ax_mu1.set_ylabel(r'$\mu_{\Psi}(k,z=0.5)$', fontsize=label_fs)

ax_g1.set_xlabel(r'$k\ [{\rm h/Mpc}]$', fontsize=label_fs)
ax_mu1.set_xlabel(r'$k\ [{\rm h/Mpc}]$', fontsize=label_fs)

for irow, z_plot in enumerate(z_plots):

    ax_g = axes[irow, 0]
    ax_mu = axes[irow, 1]

    for num, (s_str, a_str) in enumerate(
        zip(sigma_consts, alpha_consts)
    ):

        sig = skey(s_str)
        alp = akey(a_str)

        if sig not in data['tk'][z_plot]:
            continue
        if alp not in data['tk'][z_plot][sig]:
            continue
        if sig not in data['bg_data']:
            continue
        if alp not in data['bg_data'][sig]:
            continue

        tk = data['tk'][z_plot][sig][alp]
        bg = data['bg_data'][sig][alp]

        if not isinstance(tk, np.ndarray) or not isinstance(bg, np.ndarray):
            continue

        (
            a,
            H_conf,
            rho_g,
            rho_b,
            rho_cdm,
            rho_ncdm,
            rho_ur,
        ) = bg_at_z(bg, z_plot)

        delta_m, _, rho_m, _ = compute_delta_components(
            tk,
            rho_g,
            rho_b,
            rho_cdm,
            rho_ncdm,
            rho_ur,
        )

        k_Mpc, _, Delta_m = compute_Delta_m(
            tk,
            H_conf,
            delta_m,
        )

        gamma, mu_psi = compute_gamma_mu(
            tk,
            a,
            rho_m,
            Delta_m,
        )

        # Plot against the CLASS transfer-function convention k [h/Mpc].
        k_hMpc = k_Mpc / h

        color = colors[num]
        label = (
            rf'$(\sigma={float(s_str):g},\,'
            rf'\alpha={float(a_str):g})$'
        )

        ax_g.plot(
            k_hMpc,
            gamma,
            '-',
            lw=lw_f,
            c=color,
            label=label,
        )

        ax_mu.plot(
            k_hMpc,
            mu_psi,
            '-',
            lw=lw_f,
            c=color,
        )

for ax in [ax_g0, ax_mu0]:
    ax.set_ylim(0.83, 1.43)

for ax in [ax_g1, ax_mu1]:
    ax.set_ylim(0.8, 1.35)

ax_g0.legend(
    loc='upper left',
    frameon=False,
    bbox_to_anchor=(-0.015, 1.01),
    fontsize=21,
    ncol=2,
    columnspacing=0.25,
)

plt.tight_layout()

plt.savefig(
    './Figs/mu_gamma_2x2.pdf',
    format='pdf',
    dpi=300,
    bbox_inches='tight',
    pad_inches=0.1,
)

plt.show()


## 3. Linear matter and Weyl-potential spectra — paper Fig. 9

At $z=0$ we construct the linear matter power spectrum and the dimensionless
Weyl-potential spectrum from the `hi_class` transfer functions. The Weyl potential is
$
\Phi_{\rm W}=\frac{\Phi+\Psi}{2}.
$

The lower panels show ratios with respect to the separately loaded $\Lambda$CDM transfer
functions. The plotted range is the linear-theory prediction; the continuation toward
non-linear scales should therefore be interpreted as a formal linear extrapolation, as in the paper.


In [ ]:
# -------------------------------------------------
# Primordial spectrum and redshift
# -------------------------------------------------

z_plot = 0.0

A_s = 2.092e-9
n_s = 0.9626
k_pivot_Mpc = 0.05


# -------------------------------------------------
# Transfer-function column layouts
#
# The CCDE transfer files contain two additional scalar-field columns
# (vx_smg and vx_prime_smg), so phi and psi are shifted by two columns
# relative to the standard LambdaCDM CLASS output.
# -------------------------------------------------

COL_CCDE = {
    'k':      0,
    'd_b':    2,
    'd_cdm':  3,
    'd_ncdm': 5,
    'phi':   10,
    'psi':   11,
}

COL_LCDM = {
    'k':      0,
    'd_b':    2,
    'd_cdm':  3,
    'd_ncdm': 5,
    'phi':    8,
    'psi':    9,
}


# -------------------------------------------------
# Background densities
#
# These columns are the same in the CCDE and LambdaCDM
# background outputs:
#
#   10: rho_b      -> Python index 9
#   11: rho_cdm    -> Python index 10
#   12: rho_ncdm   -> Python index 11
# -------------------------------------------------

def get_bg_rhos(bg_arr, z_target):
    """Return baryon, CDM, and massive-neutrino densities."""

    z_bg = bg_arr[:, 0]
    idx = np.argmin(np.abs(z_bg - z_target))

    rho_b = bg_arr[idx, 9]
    rho_cdm = bg_arr[idx, 10]
    rho_ncdm = bg_arr[idx, 11]

    return rho_b, rho_cdm, rho_ncdm


# -------------------------------------------------
# Construct matter and Weyl-potential spectra
# -------------------------------------------------

def spectra_from_tk(
    tk_arr,
    rho_b,
    rho_cdm,
    rho_ncdm,
    col,
):
    """
    Construct the linear matter power spectrum and the dimensionless
    Weyl-potential spectrum from a transfer-function table.

    Parameters
    ----------
    tk_arr : array
        CLASS/hi_class transfer-function output.
    rho_b, rho_cdm, rho_ncdm : float
        Background densities at the selected redshift.
    col : dict
        Column map appropriate to either the CCDE or LambdaCDM
        transfer-function format.

    Notes
    -----
    The transfer-function k column is in h/Mpc. It is converted to
    physical Mpc^{-1} only when evaluating the primordial spectrum,
    whose pivot is k_p = 0.05 Mpc^{-1}.
    """

    k_hMpc = tk_arr[:, col['k']]
    k_Mpc = k_hMpc * h

    d_b = tk_arr[:, col['d_b']]
    d_cdm = tk_arr[:, col['d_cdm']]
    d_ncdm = tk_arr[:, col['d_ncdm']]

    phi = tk_arr[:, col['phi']]
    psi = tk_arr[:, col['psi']]

    # Total matter density and density contrast.
    rho_m = rho_b + rho_cdm + rho_ncdm

    delta_m = (
        rho_b * d_b
        + rho_cdm * d_cdm
        + rho_ncdm * d_ncdm
    ) / rho_m

    # Weyl potential.
    Phi_W = 0.5 * (phi + psi)

    # Dimensionless primordial curvature spectrum.
    P_R = A_s * (k_Mpc / k_pivot_Mpc)**(n_s - 1.0)

    # Linear matter power spectrum in (Mpc/h)^3.
    P_mm = (
        P_R
        * delta_m**2
        * (2.0 * np.pi**2)
        / k_hMpc**3
    )

    # Dimensionless Weyl-potential spectrum.
    P_W = P_R * Phi_W**2

    return k_hMpc, P_mm, P_W


# -------------------------------------------------
# LambdaCDM reference
# -------------------------------------------------

sig_lcdm = skey("0.0")
alp_lcdm = akey("0.0")

tk_base = data['tk'][z_plot][sig_lcdm][alp_lcdm]
bg_base = data['bg_data'][sig_lcdm][alp_lcdm]

if not isinstance(tk_base, np.ndarray):
    raise RuntimeError(
        "LambdaCDM transfer functions are required for the Fig. 9 ratios."
    )

if not isinstance(bg_base, np.ndarray):
    raise RuntimeError(
        "LambdaCDM background output is required for the Fig. 9 ratios."
    )

rho_b_base, rho_cdm_base, rho_ncdm_base = get_bg_rhos(
    bg_base,
    z_plot,
)

# IMPORTANT:
# Use the LambdaCDM transfer-function column map here.
k_base, Pmm_base, PW_base = spectra_from_tk(
    tk_base,
    rho_b_base,
    rho_cdm_base,
    rho_ncdm_base,
    COL_LCDM,
)


# -------------------------------------------------
# Interpolate the LambdaCDM reference spectra
# -------------------------------------------------

eps = 1e-40

interp_Pmm_base = interp1d(
    np.log(k_base),
    np.log(np.where(Pmm_base > 0, Pmm_base, eps)),
    kind='linear',
    bounds_error=False,
    fill_value='extrapolate',
)

interp_PW_base = interp1d(
    np.log(k_base),
    np.log(np.where(PW_base > 0, PW_base, eps)),
    kind='linear',
    bounds_error=False,
    fill_value='extrapolate',
)


# -------------------------------------------------
# Figure: spectra and LambdaCDM ratios
# -------------------------------------------------

fig = plt.figure(
    figsize=(19, 10),
    facecolor='w',
)

gs = fig.add_gridspec(
    2,
    2,
    height_ratios=[2.0, 1.5],
    wspace=0.25,
    hspace=0.05,
)

ax_Pm = fig.add_subplot(gs[0, 0])
ax_Pm_rat = fig.add_subplot(gs[1, 0], sharex=ax_Pm)

ax_PW = fig.add_subplot(gs[0, 1], sharex=ax_Pm)
ax_PW_rat = fig.add_subplot(gs[1, 1], sharex=ax_PW)

all_axes = [ax_Pm, ax_PW, ax_Pm_rat, ax_PW_rat]

for ax in all_axes:
    ax.tick_params(
        which='both',
        direction='in',
        top=True,
        right=True,
    )

    ax.grid(
        True,
        which='major',
        alpha=major_alpha,
    )

    ax.grid(
        True,
        which='minor',
        alpha=minor_alpha,
    )

    ax.minorticks_on()


# -------------------------------------------------
# Plot CCDE models
# -------------------------------------------------

for num, (sigma_l, alpha_l) in enumerate(
    zip(sigma_consts, alpha_consts)
):

    sig_key = skey(sigma_l)
    alp_key = akey(alpha_l)

    tk_arr = data['tk'][z_plot][sig_key][alp_key]
    bg_arr = data['bg_data'][sig_key][alp_key]

    if not isinstance(tk_arr, np.ndarray):
        continue

    if not isinstance(bg_arr, np.ndarray):
        continue

    rho_b, rho_cdm, rho_ncdm = get_bg_rhos(
        bg_arr,
        z_plot,
    )

    # IMPORTANT:
    # CCDE transfer files contain the two additional scalar-field
    # columns, so use the CCDE column map here.
    k_model, Pmm, PW = spectra_from_tk(
        tk_arr,
        rho_b,
        rho_cdm,
        rho_ncdm,
        COL_CCDE,
    )

    color = colors[num]

    label = (
        rf'$(\sigma={float(sigma_l):g},\,'
        rf'\alpha={float(alpha_l):g})$'
    )

    # Matter power spectrum
    ax_Pm.loglog(
        k_model,
        Pmm,
        '-',
        lw=lw_f,
        c=color,
        label=label,
    )

    # Weyl-potential spectrum
    ax_PW.loglog(
        k_model,
        PW,
        '-',
        lw=lw_f,
        c=color,
    )

    # LambdaCDM spectra evaluated at the CCDE k sampling.
    Pmm_base_here = np.exp(
        interp_Pmm_base(np.log(k_model))
    )

    PW_base_here = np.exp(
        interp_PW_base(np.log(k_model))
    )

    # Ratios to LambdaCDM
    ax_Pm_rat.semilogx(
        k_model,
        Pmm / Pmm_base_here,
        '-',
        lw=lw_f,
        c=color,
    )

    ax_PW_rat.semilogx(
        k_model,
        PW / PW_base_here,
        '-',
        lw=lw_f,
        c=color,
    )


# -------------------------------------------------
# Axis formatting
# -------------------------------------------------

for ax in all_axes:
    ax.set_xlim(1e-4, 1.0)

log_major = LogLocator(base=10.0)

log_minor = LogLocator(
    base=10.0,
    subs=np.arange(2, 10) * 0.1,
)

for ax in all_axes:

    ax.xaxis.set_major_locator(log_major)
    ax.xaxis.set_minor_locator(log_minor)
    ax.xaxis.set_major_formatter(LogFormatterMathtext())

for ax in [ax_Pm, ax_PW]:
    ax.tick_params(labelbottom=False)

for ax in [ax_Pm_rat, ax_PW_rat]:

    ax.yaxis.set_major_locator(
        MultipleLocator(0.2)
    )

    ax.yaxis.set_minor_locator(
        MultipleLocator(0.05)
    )


# -------------------------------------------------
# Axis labels
# -------------------------------------------------

ax_Pm.set_ylabel(
    r'$P_{\rm m}(k,z=0)\,[{\rm Mpc}^3h^{-3}]$',
    fontsize=label_fs,
)

ax_PW.set_ylabel(
    r'$\mathcal{P}_{\Phi_{\rm W}}(k,z=0)$',
    fontsize=label_fs,
)

ax_Pm_rat.set_ylabel(
    r'$P_{\rm m}/P_{\rm m}^{\Lambda{\rm CDM}}$',
    fontsize=label_fs,
)

ax_PW_rat.set_ylabel(
    r'$\mathcal{P}_{\Phi_{\rm W}}/'
    r'\mathcal{P}_{\Phi_{\rm W}}^{\Lambda{\rm CDM}}$',
    fontsize=label_fs,
)

ax_Pm_rat.set_xlabel(
    r'$k\ [{\rm h/Mpc}]$',
    fontsize=label_fs,
)

ax_PW_rat.set_xlabel(
    r'$k\ [{\rm h/Mpc}]$',
    fontsize=label_fs,
)


# -------------------------------------------------
# Plot ranges
# -------------------------------------------------

ax_Pm_rat.set_ylim(0.35, 1.30)
ax_PW_rat.set_ylim(0.35, 1.30)

ax_Pm.set_ylim(1, 4e4)
ax_PW.set_ylim(5e-15, 1e-9)


# -------------------------------------------------
# Force decade labels on the Weyl-spectrum axis
# -------------------------------------------------

y_minor = LogLocator(
    base=10.0,
    subs=np.arange(2, 10) * 0.1,
    numticks=1000,
)

fmt_decades = LogFormatterMathtext(
    base=10.0,
    labelOnlyBase=True,
)

ymin, ymax = ax_PW.get_ylim()

emin = int(np.floor(np.log10(ymin)))
emax = int(np.ceil(np.log10(ymax)))

ax_PW.yaxis.set_major_locator(
    FixedLocator(
        [
            10.0**e
            for e in range(emin, emax + 1)
        ]
    )
)

ax_PW.yaxis.set_minor_locator(y_minor)
ax_PW.yaxis.set_major_formatter(fmt_decades)
ax_PW.yaxis.set_minor_formatter(NullFormatter())


# -------------------------------------------------
# Legend
# -------------------------------------------------

ax_Pm.legend(
    loc='upper left',
    bbox_to_anchor=(0.0, 0.48),
    frameon=True,
    fontsize=17.8,
    ncol=2,
    columnspacing=0.2,
)


# plt.tight_layout()


# -------------------------------------------------
# Save figure
# -------------------------------------------------

plt.savefig(
    './Figs/linear_matter_Weyl_spectra_z0.pdf',
    format='pdf',
    dpi=300,
    bbox_inches='tight',
    pad_inches=0.1,
)

plt.show()

## 4. Weyl-potential spectrum at intermediate redshifts — paper Fig. 11

This figure shows the ratio of the dimensionless Weyl-potential spectrum,
$
\mathcal{P}_{\Phi_{\rm W}}(k,z)
=
\mathcal{P}_{\mathcal R}(k)
\left[\frac{\Phi(k,z)+\Psi(k,z)}{2}\right]^2,
$
to the corresponding $\Lambda$CDM spectrum at
$z=0.25,\ 0.75,\ 1,\ 2$.

The separate transfer-function column maps for CCDE and $\Lambda$CDM are retained:
the CCDE files contain two additional scalar-field columns, so the $\Phi$ and $\Psi$
columns are shifted by two positions relative to the standard $\Lambda$CDM output.


In [ ]:

# -------------------------------------------------
# Redshifts shown in paper Fig. 11
# -------------------------------------------------
z_list = [0.25, 0.75, 1.0, 2.0]


# -------------------------------------------------
# Weyl-potential spectrum
# -------------------------------------------------
def weyl_spectrum_from_tk(tk_arr, col):
    """
    Construct the dimensionless Weyl-potential spectrum.

    The transfer-function k column is in h/Mpc. It is converted to
    physical Mpc^{-1} when evaluating the primordial spectrum,
    whose pivot is k_p = 0.05 Mpc^{-1}.
    """
    k_hMpc = tk_arr[:, col['k']]
    k_Mpc = k_hMpc * h

    phi = tk_arr[:, col['phi']]
    psi = tk_arr[:, col['psi']]

    Phi_W = 0.5 * (phi + psi)

    P_R = A_s * (k_Mpc / k_pivot_Mpc)**(n_s - 1.0)
    P_W = P_R * Phi_W**2

    return k_hMpc, P_W


def get_lcdm_weyl_interpolator(z_plot):
    """Return a log-log interpolator for the LambdaCDM Weyl spectrum."""
    sig_lcdm = skey("0.0")
    alp_lcdm = akey("0.0")

    tk_base = data['tk'][z_plot][sig_lcdm][alp_lcdm]

    if not isinstance(tk_base, np.ndarray):
        raise RuntimeError(
            f"LambdaCDM transfer functions are required at z={z_plot}."
        )

    k_base, PW_base = weyl_spectrum_from_tk(
        tk_base,
        COL_LCDM,
    )

    eps = 1.e-40

    interp_base = interp1d(
        np.log(k_base),
        np.log(np.where(PW_base > 0.0, PW_base, eps)),
        kind='linear',
        bounds_error=False,
        fill_value='extrapolate',
    )

    return interp_base


# -------------------------------------------------
# Figure: 2 x 2 redshift panels
# -------------------------------------------------
fig, axes = plt.subplots(
    2,
    2,
    figsize=(16, 11),
    sharex=True,
    sharey=True,
    facecolor='w',
)

axes = axes.flatten()

plt.subplots_adjust(
    wspace=0.0,
    hspace=0.0,
)

for ax in axes:
    ax.tick_params(
        which='both',
        direction='in',
        top=True,
        right=True,
    )

    ax.grid(True, which='major', alpha=major_alpha)
    ax.grid(True, which='minor', alpha=minor_alpha)

    ax.minorticks_on()
    ax.set_xscale('log')

    ax.axhline(
        1.0,
        color='k',
        lw=1.5,
        alpha=0.8,
    )


# -------------------------------------------------
# Plot CCDE / LambdaCDM Weyl-spectrum ratios
# -------------------------------------------------
for iz, z_plot in enumerate(z_list):

    ax = axes[iz]

    print(f"Plotting Weyl-potential ratios at z = {z_plot}")

    interp_PW_base = get_lcdm_weyl_interpolator(z_plot)

    for num, (sigma_l, alpha_l) in enumerate(
        zip(sigma_consts, alpha_consts)
    ):

        sig_key = skey(sigma_l)
        alp_key = akey(alpha_l)

        tk_arr = (
            data['tk']
            .get(z_plot, {})
            .get(sig_key, {})
            .get(alp_key, None)
        )

        if not isinstance(tk_arr, np.ndarray):
            print(
                f"Skipping {sig_key}, {alp_key} "
                f"at z={z_plot}: transfer file missing."
            )
            continue

        # CCDE transfer files contain the two additional scalar-field columns.
        k_model, PW_model = weyl_spectrum_from_tk(
            tk_arr,
            COL_CCDE,
        )

        PW_base_here = np.exp(
            interp_PW_base(np.log(k_model))
        )

        ratio_W = PW_model / PW_base_here

        color = colors[num]

        label = (
            rf'$(\sigma={float(sigma_l):g},\,'
            rf'\alpha={float(alpha_l):g})$'
        )

        ax.semilogx(
            k_model,
            ratio_W,
            '-',
            lw=lw_f,
            c=color,
            label=label,
        )

    ax.text(
        0.05,
        0.08,
        rf'$z={z_plot:g}$',
        transform=ax.transAxes,
        fontsize=label_fs,
        ha='left',
        va='bottom',
        bbox=dict(
            facecolor='white',
            edgecolor='none',
            alpha=0.75,
            pad=3.0,
        ),
    )


# -------------------------------------------------
# Axis formatting
# -------------------------------------------------
for ax in axes:
    ax.set_xlim(2.e-4, 1.4)
    ax.set_ylim(0.35, 1.75)

    ax.yaxis.set_major_locator(MultipleLocator(0.2))
    ax.yaxis.set_minor_locator(MultipleLocator(0.05))

    ax.xaxis.set_major_locator(LogLocator(base=10.0))
    ax.xaxis.set_minor_locator(
        LogLocator(
            base=10.0,
            subs=np.arange(2, 10) * 0.1,
        )
    )

    ax.xaxis.set_major_formatter(LogFormatterMathtext())
    ax.xaxis.set_minor_formatter(NullFormatter())


for ax in [axes[1], axes[3]]:
    ax.tick_params(labelleft=False)

axes[0].set_ylabel(
    r'$\mathcal{P}_{\Phi_{\rm W}}/'
    r'\mathcal{P}_{\Phi_{\rm W}}^{\Lambda{\rm CDM}}$',
    fontsize=label_fs,
)

axes[2].set_ylabel(
    r'$\mathcal{P}_{\Phi_{\rm W}}/'
    r'\mathcal{P}_{\Phi_{\rm W}}^{\Lambda{\rm CDM}}$',
    fontsize=label_fs,
)

for ax in [axes[0], axes[1]]:
    ax.tick_params(labelbottom=False)

axes[2].set_xlabel(r'$k\ [{\rm h/Mpc}]$', fontsize=label_fs)
axes[3].set_xlabel(r'$k\ [{\rm h/Mpc}]$', fontsize=label_fs)


# -------------------------------------------------
# Legend
# -------------------------------------------------
legend = axes[0].legend(
    loc='upper left',
    frameon=True,
    fontsize=15.5,
    ncol=2,
    handlelength=2.2,
)

legend.get_frame().set_alpha(0.85)


plt.tight_layout()

plt.savefig(
    './Figs/Weyl_potential_ratios_redshifts.pdf',
    format='pdf',
    dpi=300,
    bbox_inches='tight',
    pad_inches=0.08,
)

plt.show()
